# 06 Gemma 4 E4B Context Pruning Experiment (Colab)

This notebook runs the context-pruning generation experiment with the recommended Gemma model.

Default run:
- model: `google/gemma-4-E4B-it`
- variant: `B_pruned_context_by_question_type`
- limit: 5 questions first

If the 5-question test is stable, change `RUN_LIMIT = 0` to run all rows in the wrong-30 eval CSV.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/beomsookim1020/chatbot.git'
BRANCH = 'colab-generation'
PROJECT_DIR = Path('/content/chatbot')

DRIVE_INPUT_ROOT = Path('/content/drive/MyDrive/chatbot_colab_inputs')
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/chatbot_colab_outputs')

PREDICTION_REL = Path('outputs/predictions/best_variant_predictions.jsonl')
EVAL_REL = Path('data/eval/representative_wrong_30_eval_batch_format.csv')
CHUNK_REL = Path('indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl')
SOURCE_STORE_REL = Path('data/source_store_v2_690.jsonl')

MODEL_NAME = 'google/gemma-4-E4B-it'
FALLBACK_MODEL_NAME = 'google/gemma-4-E2B-it'
MAX_NEW_TOKENS = 384
RUN_LIMIT = 5  # Use 0 for the full wrong-30 eval CSV.
RUN_VARIANTS = ['B_pruned_context_by_question_type']

RUN_CONTEXT_ONLY_DRY_RUN = True
CONTEXT_ONLY_DRY_RUN_LIMIT = 2
RUN_GENERATION = True

LOCAL_OUTPUT_ROOT = PROJECT_DIR / 'outputs/gemma_context_experiments'
DRIVE_EXPERIMENT_ROOT = DRIVE_OUTPUT_ROOT / 'gemma_context_experiments'
DRY_RUN_NAME = 'gemma4_e4b_dry_run'
GENERATION_RUN_NAME = 'gemma4_e4b_b_pruned'

print('branch:', BRANCH)
print('model:', MODEL_NAME)
print('fallback:', FALLBACK_MODEL_NAME)
print('eval:', EVAL_REL)
print('predictions:', PREDICTION_REL)
print('chunks:', CHUNK_REL)
print('source_store:', SOURCE_STORE_REL)
print('run_limit:', RUN_LIMIT)
print('variants:', RUN_VARIANTS)

## 1. Check GPU

In [ ]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Colab GPU is not enabled. Select Runtime > Change runtime type > GPU.')

!nvidia-smi

## 2. Clone or pull colab-generation

In [ ]:
import os
import subprocess

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

os.chdir(PROJECT_DIR)
print('cwd:', Path.cwd())
subprocess.run(['git', 'status', '--short'], check=True)

## 3. Install dependencies

Gemma 4 may need a recent Transformers version, so this cell updates the HF stack.

In [ ]:
%pip install -q -r requirements.txt
%pip install -q -U transformers accelerate sentencepiece protobuf huggingface_hub

## 4. Hugging Face login and model access check

If the model is gated, accept the model license on Hugging Face and add `HF_TOKEN` to Colab Secrets.

In [ ]:
import os

hf_token = os.environ.get('HF_TOKEN')
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN') or hf_token
except Exception:
    pass

if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print('HF_TOKEN login complete')
else:
    print('HF_TOKEN not found. If model access fails, add HF_TOKEN to Colab Secrets or run huggingface_hub.login().')

from transformers import AutoConfig

try:
    cfg = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
    print('model access ok:', MODEL_NAME)
    print('model_type:', getattr(cfg, 'model_type', 'unknown'))
except Exception as exc:
    print('model access failed:', MODEL_NAME)
    print(type(exc).__name__, exc)
    print('Fallback option:', FALLBACK_MODEL_NAME)
    raise

## 5. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
print('Drive input root:', DRIVE_INPUT_ROOT)
print('Drive output root:', DRIVE_OUTPUT_ROOT)

## 6. Validate and copy input files

In [ ]:
import shutil

required_inputs = [
    ('eval', DRIVE_INPUT_ROOT / EVAL_REL),
    ('predictions', DRIVE_INPUT_ROOT / PREDICTION_REL),
    ('chunks', DRIVE_INPUT_ROOT / CHUNK_REL),
    ('source_store', DRIVE_INPUT_ROOT / SOURCE_STORE_REL),
]
missing = [(name, path) for name, path in required_inputs if not path.exists()]
if missing:
    detail = '
'.join(f'- {name}: {path}' for name, path in missing)
    raise FileNotFoundError('Missing Drive input file(s):
' + detail)

copy_pairs = [
    (DRIVE_INPUT_ROOT / EVAL_REL, PROJECT_DIR / EVAL_REL),
    (DRIVE_INPUT_ROOT / PREDICTION_REL, PROJECT_DIR / PREDICTION_REL),
    (DRIVE_INPUT_ROOT / CHUNK_REL, PROJECT_DIR / CHUNK_REL),
    (DRIVE_INPUT_ROOT / SOURCE_STORE_REL, PROJECT_DIR / SOURCE_STORE_REL),
]
for src, dst in copy_pairs:
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f'copied: {src} -> {dst} ({dst.stat().st_size:,} bytes)')

script_path = PROJECT_DIR / 'experiments/context_pruning_experiment.py'
if not script_path.exists():
    raise FileNotFoundError(f'Experiment script is missing. Push/pull colab-generation first: {script_path}')
print('script:', script_path)

## 7. Runner helper

In [ ]:
import sys

CREATED_OUTPUT_DIRS = []

def list_experiment_outputs():
    LOCAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    return {path.resolve() for path in LOCAL_OUTPUT_ROOT.iterdir() if path.is_dir()}

def run_gemma_experiment(*, context_only: bool, limit: int, run_name: str):
    before = list_experiment_outputs()
    cmd = [
        sys.executable,
        str(PROJECT_DIR / 'experiments/context_pruning_experiment.py'),
        '--predictions', str(PREDICTION_REL),
        '--eval-csv', str(EVAL_REL),
        '--chunks', str(CHUNK_REL),
        '--source-store', str(SOURCE_STORE_REL),
        '--output-root', str(LOCAL_OUTPUT_ROOT.relative_to(PROJECT_DIR)),
        '--run-name', run_name,
        '--model-name', MODEL_NAME,
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--limit', str(limit),
    ]
    if context_only:
        cmd.append('--context-only')
    for variant in RUN_VARIANTS:
        cmd.extend(['--variant', variant])
    print('Running command:')
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT_DIR, check=True)
    after = list_experiment_outputs()
    created = sorted(after - before, key=lambda path: path.stat().st_mtime)
    if not created:
        raise RuntimeError('Could not find the new output directory.')
    CREATED_OUTPUT_DIRS.extend(created)
    print('created output:', created[-1])
    return created[-1]

## 8. Context-only dry run

This validates context construction without loading the model.

In [ ]:
if RUN_CONTEXT_ONLY_DRY_RUN:
    dry_output_dir = run_gemma_experiment(
        context_only=True,
        limit=CONTEXT_ONLY_DRY_RUN_LIMIT,
        run_name=DRY_RUN_NAME,
    )
else:
    dry_output_dir = None
    print('Context-only dry run skipped.')

## 9. Run Gemma generation

The default run is only 5 questions. Use `RUN_LIMIT = 0` for the full wrong-30 eval CSV after the smoke test is stable.

In [ ]:
if RUN_GENERATION:
    generation_output_dir = run_gemma_experiment(
        context_only=False,
        limit=RUN_LIMIT,
        run_name=GENERATION_RUN_NAME,
    )
else:
    generation_output_dir = None
    print('Generation skipped.')

## 10. Inspect results

In [ ]:
import csv
import html as html_lib
from IPython.display import HTML, display

latest_output_dir = generation_output_dir or dry_output_dir
if latest_output_dir is None:
    raise RuntimeError('No output directory to inspect.')

metrics_path = latest_output_dir / 'context_pruning_metrics.csv'
review_path = latest_output_dir / 'context_pruning_review.csv'
summary_path = latest_output_dir / 'context_pruning_summary.md'
results_path = latest_output_dir / 'context_pruning_results.jsonl'

print('results:', results_path)
print('review:', review_path)
print('metrics:', metrics_path)
print('summary:', summary_path)

def read_csv_preview(path, limit=None):
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        rows = []
        for idx, row in enumerate(reader):
            if limit is not None and idx >= limit:
                break
            rows.append(row)
    return rows

def display_rows(rows, title, max_cols=12):
    print(f'{title}: {len(rows)} row(s) shown')
    if not rows:
        return
    columns = list(rows[0].keys())[:max_cols]
    header = ''.join(f'<th>{html_lib.escape(col)}</th>' for col in columns)
    body = []
    for row in rows:
        cells = ''.join(
            '<td style="max-width:260px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis">'
            + html_lib.escape(str(row.get(col, ''))) + '</td>'
            for col in columns
        )
        body.append(f'<tr>{cells}</tr>')
    display(HTML(
        '<div style="overflow:auto;max-height:420px">'
        f'<table border="1" style="border-collapse:collapse;font-size:12px">'
        f'<thead><tr>{header}</tr></thead><tbody>{"".join(body)}</tbody></table>'
        '</div>'
    ))

metrics_rows = read_csv_preview(metrics_path)
review_rows = read_csv_preview(review_path, limit=10)
display_rows(metrics_rows, 'metrics')
display_rows(review_rows, 'review preview')

## 11. Copy outputs to Drive

In [ ]:
DRIVE_EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

copied_dirs = []
for local_dir in CREATED_OUTPUT_DIRS:
    dst = DRIVE_EXPERIMENT_ROOT / local_dir.name
    if dst.exists():
        raise FileExistsError(f'Drive output directory already exists. Not overwriting: {dst}')
    shutil.copytree(local_dir, dst)
    copied_dirs.append(dst)
    print('copied output to Drive:', dst)

if not copied_dirs:
    print('No new output directory to copy.')
else:
    print('Drive output dirs:')
    for path in copied_dirs:
        print('-', path)

## Review files

- `context_pruning_review.csv`: manual review file with `manual_correct`, `failure_type`, `review_note`
- `context_pruning_metrics.csv`: automatic proxy metrics for the Gemma run
- `context_pruning_summary.md`: summary and failure examples
- `context_pruning_results.jsonl`: detailed generation records with `used_context`

After the 5-question smoke test is stable, change `RUN_LIMIT = 0` to run the full wrong-30 eval CSV.